<a href="https://colab.research.google.com/github/chaitra0312/ML/blob/main/Amazon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from tqdm import tqdm
train_df = pd.read_csv("/content/train.csv", engine='python', on_bad_lines='warn')
tqdm.pandas()
train_df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [ ]:
# 1️⃣ Brand name extraction
def extract_brand(text):
    if not isinstance(text, str):
        return None

    # Extract part after 'Item Name:'
    match = re.search(r'Item Name:\s*([^\n\r,]+)', text)
    if match:
        brand_part = match.group(1).strip()
        # Split and take the first word if it exists
        parts = brand_part.split()
        if len(parts) > 0:
            return parts[0]
    return None

# 2️⃣ Product category (like Sauce, Cookies, Powder)
def extract_category(text):
    if not isinstance(text, str):
        return None

    # Get only the first line (usually item name)
    line = text.split('\n')[0]

    # Try to find a likely product category (last meaningful word before a comma or bracket)
    parts = re.split(r'[,:()]', line)
    parts = [p.strip() for p in parts if p.strip()]  # remove empty parts

    if len(parts) == 0:
        return None

    # Take the last part and last word from it (likely to be category name)
    last_part = parts[-1]
    words = last_part.split()

    if len(words) == 0:
        return None

    # Example: "La Victoria Green Taco Sauce Mild" → last word "Mild" isn’t category, so try second-last
    if len(words) >= 2 and words[-1].lower() in ["mild", "pack", "fl", "oz", "ounce"]:
        return words[-2]
    return words[-1]


# 3️⃣ Pack size
def extract_pack_size(text):
    if not isinstance(text, str):
        return 1
    match = re.search(r'[Pp]ack of (\d+)', text)
    return int(match.group(1)) if match else 1

# 4️⃣ Weight and unit
def extract_weight(text):
    if not isinstance(text, str):
        return None
    match = re.search(r'(\d+\.?\d*)\s?(Ounce|Oz|Fl Oz|Gram|g|ml|Lb|Pound)', text, re.IGNORECASE)
    return float(match.group(1)) if match else None

def extract_unit(text):
    if not isinstance(text, str):
        return None
    match = re.search(r'Unit:\s*([A-Za-z ]+)', text)
    if match:
        return match.group(1).strip().lower()
    match2 = re.search(r'(\d+\.?\d*)\s?(Ounce|Oz|Fl Oz|Gram|g|ml|Lb|Pound)', text, re.IGNORECASE)
    return match2.group(2).lower() if match2 else None

# 5️⃣ Value
def extract_value(text):
    if not isinstance(text, str):
        return None
    match = re.search(r'Value:\s*([\d\.]+)', text)
    return float(match.group(1)) if match else None

In [ ]:
train_df['brand'] = train_df['catalog_content'].progress_apply(extract_brand)
train_df['category'] = train_df['catalog_content'].progress_apply(extract_category)
train_df['pack_size'] = train_df['catalog_content'].progress_apply(extract_pack_size)
train_df['weight'] = train_df['catalog_content'].progress_apply(extract_weight)
train_df['unit'] = train_df['catalog_content'].progress_apply(extract_unit)
train_df['value'] = train_df['catalog_content'].progress_apply(extract_value)


100%|██████████| 48159/48159 [00:00<00:00, 416368.95it/s]


In [ ]:
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2',device='cuda' if torch.cuda.is_available() else 'cpu')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = text.strip()
    return text

train_df['clean_text'] = train_df['catalog_content'].apply(preprocess_text)


In [ ]:
# Encode in batches to avoid memory issues
batch_size = 256
embeddings = []

for i in range(0, len(train_df), batch_size):
    batch_texts = train_df['clean_text'].iloc[i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch_texts, batch_size=batch_size, show_progress_bar=True)
    embeddings.extend(batch_embeddings)

# Convert to numpy array
text_embeddings = np.array(embeddings)
print("Text embedding shape:", text_embeddings.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Text embedding shape: (48159, 384)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode categorical variables
label_cols = ['brand', 'category', 'unit']
for col in label_cols:
    le = LabelEncoder()
    train_df[col] = train_df[col].astype(str).fillna("Unknown")
    train_df[col] = le.fit_transform(train_df[col])

# Select numeric features
numeric_features = train_df[['pack_size', 'weight', 'value', 'brand', 'category', 'unit']].fillna(0)

# Normalize numeric features
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

# Combine numeric + BERT embeddings
X = np.hstack((numeric_scaled, text_embeddings))
y = train_df['price'].values
print("Final feature shape:", X.shape)


Final feature shape: (48159, 390)


In [ ]:
from sklearn.model_selection import train_test_split

# X = your combined embeddings + numeric data
# y = your target column (price)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='gpu_hist'  # Use GPU for faster training
)


In [ ]:
xgb_model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42
)


In [ ]:
y_train_log = np.log1p(y_train)  # log(1 + price)
y_val_log = np.log1p(y_val)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Make sure the model is fitted before predicting
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("MAE:", mae)
print("R² Score:", r2)

MAE: 13.14587604620924
R² Score: 0.3437714229748249


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [03:50:05] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import clip
from PIL import Image
import torch
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, preprocess = clip.load("ViT-B/32", device=device)


100%|███████████████████████████████████████| 338M/338M [00:06<00:00, 52.6MiB/s]


In [ ]:
def get_image_embedding(url):
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img_input = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = model_clip.encode_image(img_input)
        embedding = embedding.cpu().numpy().flatten()
        return embedding
    except:
        # if image fails to load, return zeros
        return np.zeros(model_clip.visual.output_dim)


In [ ]:
import numpy as np
from tqdm import tqdm

image_embeddings = []

for url in tqdm(train_df['image_link'], total=len(train_df)):
    emb = get_image_embedding(url)
    image_embeddings.append(emb)

image_embeddings = np.array(image_embeddings)
print("Image embeddings shape:", image_embeddings.shape)


  1%|          | 485/48159 [00:52<1:19:26, 10.00it/s]

In [ ]:
# numeric features
numeric_features = train_df[['pack_size', 'weight', 'value', 'brand', 'category', 'unit']].fillna(0)
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

# concatenate all
X_final = np.hstack((numeric_scaled, text_embeddings, image_embeddings))
y_final = train_df['price'].values
print("Final feature shape:", X_final.shape)


Final feature shape: (75000, 902)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='gpu_hist',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("MAE:", mae)
print("R² Score:", r2)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:24:13] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [12:25:49] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:


MAE: 12.911254800445555
R² Score: 0.3051967997838997


In [ ]:
# Apply log1p to handle skewed prices
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)


In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',  # updated for GPU warning
    device='cuda',       # use GPU
    random_state=42
)

# Train model
xgb_model.fit(X_train, y_train_log)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred_log = xgb_model.predict(X_val)

# Convert back to original scale
y_pred = np.expm1(y_pred_log)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("MAE after log-transform:", mae)
print("R² Score after log-transform:", r2)


MAE after log-transform: 12.124391289255142
R² Score after log-transform: 0.221406081010874


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X_final, dtype=torch.float32)
y_tensor = torch.tensor(y_final, dtype=torch.float32).view(-1, 1)

# Train-validation split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
class PricePredictorNN(nn.Module):
    def __init__(self, input_dim):
        super(PricePredictorNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # single output for price
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_final.shape[1]
model = PricePredictorNN(input_dim).to(device)


In [ ]:
criterion = nn.MSELoss()  # Mean Squared Error
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch 1/20, Train Loss: 810.6593, Val Loss: 1181.8574
Epoch 2/20, Train Loss: 682.7104, Val Loss: 1207.8887
Epoch 3/20, Train Loss: 624.0965, Val Loss: 1156.5154
Epoch 4/20, Train Loss: 582.1271, Val Loss: 1615.7157
Epoch 5/20, Train Loss: 544.2836, Val Loss: 3669.0171
Epoch 6/20, Train Loss: 496.2547, Val Loss: 18351350.1225
Epoch 7/20, Train Loss: 469.3943, Val Loss: 1388198.3255
Epoch 8/20, Train Loss: 466.5568, Val Loss: 5290465.8614
Epoch 9/20, Train Loss: 421.3772, Val Loss: 197416.9361
Epoch 10/20, Train Loss: 377.7482, Val Loss: 79641561.9840
Epoch 11/20, Train Loss: 355.1985, Val Loss: 1242625060.4629
Epoch 12/20, Train Loss: 350.8521, Val Loss: 31321734.3362
Epoch 13/20, Train Loss: 318.1642, Val Loss: 406298.6111
Epoch 14/20, Train Loss: 305.9336, Val Loss: 4162850.6489
Epoch 15/20, Train Loss: 300.1137, Val Loss: 1708357.9642
Epoch 16/20, Train Loss: 295.6250, Val Loss: 200540539.1694
Epoch 17/20, Train Loss: 266.4764, Val Loss: 1065059717.1169
Epoch 18/20, Train Loss: 281.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

model.eval()
with torch.no_grad():
    y_val_pred = model(X_val.to(device)).cpu().numpy()

mae = mean_absolute_error(y_val.numpy(), y_val_pred)
r2 = r2_score(y_val.numpy(), y_val_pred)
print("NN MAE:", mae)
print("NN R²:", r2)


NN MAE: 513.9495239257812
NN R²: -470836.28125


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)


In [ ]:
import numpy as np

y_log = np.log1p(y_final)  # log(1 + price) to reduce skew
y_tensor = torch.tensor(y_log, dtype=torch.float32).view(-1,1)


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

X_train, X_val, y_train, y_val = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
import torch.nn as nn
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

class PricePredictorNN(nn.Module):
    def __init__(self, input_dim):
        super(PricePredictorNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # output for log(price)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_scaled.shape[1]
model = PricePredictorNN(input_dim).to(device)


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)  # smaller LR


In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        # Clip gradients to avoid exploding
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch 1/20, Train Loss: 0.8734, Val Loss: 0.5559
Epoch 2/20, Train Loss: 0.4736, Val Loss: 0.5470
Epoch 3/20, Train Loss: 0.4057, Val Loss: 0.5263
Epoch 4/20, Train Loss: 0.3518, Val Loss: 0.5398
Epoch 5/20, Train Loss: 0.3003, Val Loss: 0.5341
Epoch 6/20, Train Loss: 0.2592, Val Loss: 0.5457
Epoch 7/20, Train Loss: 0.2233, Val Loss: 0.5566
Epoch 8/20, Train Loss: 0.1966, Val Loss: 0.5617
Epoch 9/20, Train Loss: 0.1741, Val Loss: 0.5697
Epoch 10/20, Train Loss: 0.1571, Val Loss: 0.5839
Epoch 11/20, Train Loss: 0.1424, Val Loss: 0.5841
Epoch 12/20, Train Loss: 0.1295, Val Loss: 0.5750
Epoch 13/20, Train Loss: 0.1213, Val Loss: 0.5848
Epoch 14/20, Train Loss: 0.1128, Val Loss: 0.5777
Epoch 15/20, Train Loss: 0.1056, Val Loss: 0.5924
Epoch 16/20, Train Loss: 0.0984, Val Loss: 0.5921
Epoch 17/20, Train Loss: 0.0939, Val Loss: 0.5862
Epoch 18/20, Train Loss: 0.0901, Val Loss: 0.5980
Epoch 19/20, Train Loss: 0.0860, Val Loss: 0.5835
Epoch 20/20, Train Loss: 0.0816, Val Loss: 0.5871


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

model.eval()
with torch.no_grad():
    y_val_pred_log = model(X_val.to(device)).cpu().numpy()
    y_val_pred = np.expm1(y_val_pred_log)  # convert back to original price

mae = mean_absolute_error(np.expm1(y_val.numpy()), y_val_pred)
r2 = r2_score(np.expm1(y_val.numpy()), y_val_pred)

print("NN MAE:", mae)
print("NN R²:", r2)


NN MAE: 13.189581871032715
NN R²: 0.24898463487625122


In [ ]:
import numpy as np

def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

# Compute SMAPE
smape_val = smape(np.expm1(y_val.numpy()), y_val_pred)  # if y_val was log-transformed
print("SMAPE on validation set:", smape_val, "%")


SMAPE on validation set: 56.95431 %


In [ ]:
# Adjust based on min/max prices in train set
y_val_pred_clipped = np.clip(y_val_pred, 0.5, 200)

# Recalculate SMAPE
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

smape_val = smape(np.expm1(y_val.numpy()), y_val_pred_clipped)
print("SMAPE after clipping:", smape_val, "%")


SMAPE after clipping: 56.927216 %


In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1,1)

batch_size = 128
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)


In [ ]:
import torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"

class PriceNN(nn.Module):
    def __init__(self, input_dim):
        super(PriceNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.model(x)

model = PriceNN(X_scaled.shape[1]).to(device)


In [ ]:
criterion = nn.MSELoss()  # log-target MSE
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch 1/30, Train Loss: 1.2345, Val Loss: 0.4811
Epoch 2/30, Train Loss: 0.4486, Val Loss: 0.4404
Epoch 3/30, Train Loss: 0.4045, Val Loss: 0.4258
Epoch 4/30, Train Loss: 0.3761, Val Loss: 0.4229
Epoch 5/30, Train Loss: 0.3520, Val Loss: 0.4221
Epoch 6/30, Train Loss: 0.3287, Val Loss: 0.4211
Epoch 7/30, Train Loss: 0.3108, Val Loss: 0.4238
Epoch 8/30, Train Loss: 0.2963, Val Loss: 0.4293
Epoch 9/30, Train Loss: 0.2788, Val Loss: 0.4273
Epoch 10/30, Train Loss: 0.2658, Val Loss: 0.4329
Epoch 11/30, Train Loss: 0.2556, Val Loss: 0.4258
Epoch 12/30, Train Loss: 0.2445, Val Loss: 0.4374
Epoch 13/30, Train Loss: 0.2359, Val Loss: 0.4333
Epoch 14/30, Train Loss: 0.2262, Val Loss: 0.4342
Epoch 15/30, Train Loss: 0.2185, Val Loss: 0.4374
Epoch 16/30, Train Loss: 0.2132, Val Loss: 0.4356
Epoch 17/30, Train Loss: 0.2075, Val Loss: 0.4366
Epoch 18/30, Train Loss: 0.2003, Val Loss: 0.4385
Epoch 19/30, Train Loss: 0.1953, Val Loss: 0.4364
Epoch 20/30, Train Loss: 0.1908, Val Loss: 0.4359
Epoch 21/

In [ ]:
def smape_safe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    # Avoid division by zero
    denominator = np.where(denominator == 0, 1, denominator)
    return 100 * np.mean(np.abs(y_pred - y_true) / denominator)


In [ ]:
print(y_val_pred.shape, y_val_tensor.shape)


(15000, 1) torch.Size([15000, 1])


In [ ]:
y_val_true = y_val_tensor.cpu().numpy().flatten()
y_val_pred = y_val_pred.flatten()  # already exponentiated back to original scale


In [ ]:
y_val_pred = np.clip(y_val_pred, 0.01, 1000)  # adjust max based on dataset


In [ ]:

def smape_safe(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1, denominator)  # avoid division by 0
    return 100 * np.mean(np.abs(y_pred - y_true) / denominator)

smape_val = smape_safe(y_val_true, y_val_pred)
print("Validation SMAPE:", smape_val, "%")



Validation SMAPE: 132.2692 %


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import xgboost as xgb


In [ ]:
# Numeric/tabular features
train_df['unit_price'] = train_df['value'] / train_df['pack_size'].replace(0,1)
train_df['brand_mean_price'] = train_df.groupby('brand')['price'].transform('mean')
train_df['category_mean_price'] = train_df.groupby('category')['price'].transform('mean')

numeric_cols = ['value', 'pack_size', 'unit_price', 'brand_mean_price', 'category_mean_price']
X_numeric = train_df[numeric_cols].fillna(0).values
y = np.log1p(train_df['price'].values)  # log-transform target

# Split into train/validation
X_num_train, X_num_val, y_train, y_val = train_test_split(X_numeric, y, test_size=0.2, random_state=42)

# Split text and image embeddings
X_text_train, X_text_val, _, _ = train_test_split(text_embeddings, y, test_size=0.2, random_state=42)
X_image_train, X_image_val, _, _ = train_test_split(image_embeddings, y, test_size=0.2, random_state=42)

In [ ]:
dtrain = xgb.DMatrix(X_num_train, label=y_train)
dval = xgb.DMatrix(X_num_val, label=y_val)

params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',   # works on CPU/GPU
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

xgb_model = xgb.train(params, dtrain, num_boost_round=300, evals=[(dval,'val')], early_stopping_rounds=20, verbose_eval=50)

# Predict XGBoost
y_val_pred_xgb = np.expm1(xgb_model.predict(dval))


[0]	val-rmse:0.93831
[50]	val-rmse:0.66268
[100]	val-rmse:0.65214
[150]	val-rmse:0.64957
[200]	val-rmse:0.64720
[250]	val-rmse:0.64555
[299]	val-rmse:0.64452


In [ ]:
# Assume text embeddings (BERT) and image embeddings (CLIP) are ready
# X_text_train, X_text_val, X_image_train, X_image_val

# Only numeric features
X_train_nn = X_num_train
X_val_nn = X_num_val

# No BERT or CLIP embeddings included
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(X_train_nn)
X_val_nn_scaled = scaler.transform(X_val_nn)

X_train_tensor = torch.tensor(X_train_nn_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_nn_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1,1)

batch_size = 128
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)


In [ ]:
class PriceNN(nn.Module):
    def __init__(self, input_dim):
        super(PriceNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.model(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
nn_model = PriceNN(X_train_nn_scaled.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.0003)


In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    nn_model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = nn_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nn_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    nn_model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = nn_model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch 1/30, Train Loss: 1.2166, Val Loss: 0.5191
Epoch 2/30, Train Loss: 0.5014, Val Loss: 0.5872
Epoch 3/30, Train Loss: 0.4942, Val Loss: 0.4951
Epoch 4/30, Train Loss: 0.4872, Val Loss: 0.5071
Epoch 5/30, Train Loss: 0.4841, Val Loss: 0.4765
Epoch 6/30, Train Loss: 0.4808, Val Loss: 0.4903
Epoch 7/30, Train Loss: 0.4799, Val Loss: 0.4713
Epoch 8/30, Train Loss: 0.4773, Val Loss: 0.4820
Epoch 9/30, Train Loss: 0.4767, Val Loss: 0.4613
Epoch 10/30, Train Loss: 0.4744, Val Loss: 0.4767
Epoch 11/30, Train Loss: 0.4727, Val Loss: 0.4964
Epoch 12/30, Train Loss: 0.4709, Val Loss: 0.4753
Epoch 13/30, Train Loss: 0.4694, Val Loss: 0.4568
Epoch 14/30, Train Loss: 0.4687, Val Loss: 0.4783
Epoch 15/30, Train Loss: 0.4684, Val Loss: 0.4784
Epoch 16/30, Train Loss: 0.4687, Val Loss: 0.4685
Epoch 17/30, Train Loss: 0.4669, Val Loss: 0.4680
Epoch 18/30, Train Loss: 0.4654, Val Loss: 0.4662
Epoch 19/30, Train Loss: 0.4659, Val Loss: 0.4612
Epoch 20/30, Train Loss: 0.4644, Val Loss: 0.4640
Epoch 21/

In [ ]:
nn_model.eval()
with torch.no_grad():
    y_val_pred_log_nn = nn_model(X_val_tensor.to(device)).cpu().numpy()
    y_val_pred_nn = np.expm1(y_val_pred_log_nn)
    y_val_pred_nn = np.clip(y_val_pred_nn, 0.01, 1000)


In [ ]:
y_val_ensemble = 0.5 * y_val_pred_xgb + 0.5 * y_val_pred_nn

def smape_safe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1, denominator)
    return 100 * np.mean(np.abs(y_pred - y_true) / denominator)

y_val_true = np.expm1(y_val_tensor.numpy().flatten())
smape_val = smape_safe(y_val_true, y_val_ensemble)
print("Validation SMAPE (Ensemble):", smape_val, "%")


Validation SMAPE (Ensemble): 63.09315 %


In [ ]:
!pip install sentence-transformers


In [ ]:
# Load the model (fast + accurate)
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert text into embeddings
X_text_train = text_model.encode(train_df['catalog_content'].astype(str).tolist(), show_progress_bar=True)
# Create a dummy val_df for demonstration; a proper split should be done earlier
# In a real scenario, you would split train_df into train and validation sets
val_df = train_df.sample(frac=0.2, random_state=42)
X_text_val = text_model.encode(val_df['catalog_content'].astype(str).tolist(), show_progress_bar=True)

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Assume X_num_train, X_num_val, y_train, y_val are already split in cell aAf36slfWT3p
# We need the train_index and val_index from that split to generate text embeddings for the splits

# Re-split the data to get the indices
from sklearn.model_selection import train_test_split
# Assuming X_numeric was created earlier in aAf36slfWT3p
X_numeric = train_df[numeric_cols].fillna(0).values
y = np.log1p(train_df['price'].values)

# Perform the split again to get the indices
_, _, _, _, train_index, val_index = train_test_split(
    X_numeric, y, train_df.index, test_size=0.2, random_state=42
)


# Load the text model
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate text embeddings specifically for the training and validation sets
train_texts = train_df.loc[train_index, 'catalog_content'].astype(str).tolist()
val_texts = train_df.loc[val_index, 'catalog_content'].astype(str).tolist()

X_text_train = text_model.encode(train_texts, show_progress_bar=True)
X_text_val = text_model.encode(val_texts, show_progress_bar=True)


# Combine numeric and text features for training the first model
X_train_combined = np.hstack([X_num_train, X_text_train])
X_val_combined = np.hstack([X_num_val, X_text_val])

print("Shape of X_train_combined:", X_train_combined.shape)
print("Shape of X_val_combined:", X_val_combined.shape)

Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Shape of X_train_combined: (60000, 389)
Shape of X_val_combined: (15000, 389)


In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train_combined, label=y_train)
dval = xgb.DMatrix(X_val_combined, label=y_val)

params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.03,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'rmse'
}

evals = [(dtrain, 'train'), (dval, 'val')]

xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

# Predict
y_val_pred = np.expm1(xgb_model.predict(dval))


[0]	train-rmse:0.92512	val-rmse:0.93794
[50]	train-rmse:0.62855	val-rmse:0.67084
[100]	train-rmse:0.56265	val-rmse:0.63546
[150]	train-rmse:0.52320	val-rmse:0.62542
[200]	train-rmse:0.49321	val-rmse:0.62111
[250]	train-rmse:0.46993	val-rmse:0.61803
[300]	train-rmse:0.44921	val-rmse:0.61624
[350]	train-rmse:0.43001	val-rmse:0.61480
[400]	train-rmse:0.41226	val-rmse:0.61358
[450]	train-rmse:0.39523	val-rmse:0.61265
[500]	train-rmse:0.37867	val-rmse:0.61158
[550]	train-rmse:0.36318	val-rmse:0.61075
[600]	train-rmse:0.35000	val-rmse:0.60997
[650]	train-rmse:0.33679	val-rmse:0.60949
[700]	train-rmse:0.32402	val-rmse:0.60918
[750]	train-rmse:0.31124	val-rmse:0.60869
[800]	train-rmse:0.29959	val-rmse:0.60837
[850]	train-rmse:0.28843	val-rmse:0.60816
[900]	train-rmse:0.27758	val-rmse:0.60779
[950]	train-rmse:0.26740	val-rmse:0.60739
[999]	train-rmse:0.25680	val-rmse:0.60721


In [ ]:
import xgboost as xgb

# Wrap numeric + text features into DMatrix
dtrain_combined = xgb.DMatrix(X_train_combined, label=y_train)
dval_combined   = xgb.DMatrix(X_val_combined, label=y_val)

# Train the model
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.03,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'rmse'
}

evals = [(dtrain_combined, 'train'), (dval_combined, 'val')]

xgb_model = xgb.train(
    params,
    dtrain_combined,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)


[0]	train-rmse:0.92512	val-rmse:0.93794
[50]	train-rmse:0.62855	val-rmse:0.67084
[100]	train-rmse:0.56265	val-rmse:0.63546
[150]	train-rmse:0.52320	val-rmse:0.62542
[200]	train-rmse:0.49321	val-rmse:0.62111
[250]	train-rmse:0.46993	val-rmse:0.61803
[300]	train-rmse:0.44921	val-rmse:0.61624
[350]	train-rmse:0.43001	val-rmse:0.61480
[400]	train-rmse:0.41226	val-rmse:0.61358
[450]	train-rmse:0.39523	val-rmse:0.61265
[500]	train-rmse:0.37867	val-rmse:0.61158
[550]	train-rmse:0.36318	val-rmse:0.61075
[600]	train-rmse:0.35000	val-rmse:0.60997
[650]	train-rmse:0.33679	val-rmse:0.60949
[700]	train-rmse:0.32402	val-rmse:0.60918
[750]	train-rmse:0.31124	val-rmse:0.60869
[800]	train-rmse:0.29959	val-rmse:0.60837
[850]	train-rmse:0.28843	val-rmse:0.60816
[900]	train-rmse:0.27758	val-rmse:0.60779
[950]	train-rmse:0.26740	val-rmse:0.60739
[999]	train-rmse:0.25680	val-rmse:0.60721


In [ ]:
# Prediction is on log scale
y_pred_val_log = xgb_model.predict(dval_combined)

# Convert back to original scale
y_pred_val = np.expm1(y_pred_val_log)
y_val_orig = np.expm1(y_val)  # if y was log-transformed


In [ ]:
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1, denominator)  # avoid division by zero
    return 100 * np.mean(np.abs(y_pred - y_true) / denominator)

smape_val = smape(y_val_orig, y_pred_val)
print("Validation SMAPE:", smape_val, "%")


Validation SMAPE: 46.33487493448031 %
